In [1]:
import numpy as np
import scipy as sp
import pandas as pd

In [2]:
## Returns the magnetic moment for a given dipolar length
def mag_moment_from_dipole_length(a_dd: float, m: float) -> float:
    mu_0 = sp.constants.mu_0
    hbar = sp.constants.hbar
    return np.sqrt((12 * np.pi * a_dd * hbar ** 2) / (m * mu_0))


## Calculates the experimental parameters and returns them as a dictionary.
## Expects you pass some known parameters about the system.
def calculate_experimental_parameters(
    particle_number: int,
    dimensionless_length: float,
    lattice_wavelength: float,
    atom_mass: float,
    atom_mag_moment: float,
    external_field_ratio: float,
    external_field_height_ratio: float,
) -> dict:
    # magnetic vacuum permittivity
    mu_0 = sp.constants.mu_0
    # plack's constant
    h = sp.constants.h
    # reduced plank's constant
    hbar = sp.constants.hbar
    # bohr radius
    a_0 = sp.constants.physical_constants["Bohr radius"][0]
    # bohr magneton

    N = particle_number
    q = dimensionless_length
    lmbd = lattice_wavelength
    m = atom_mass
    p = atom_mag_moment
    nu_bar = external_field_ratio
    z_bar = external_field_height_ratio

    # lattice length
    L = lmbd / 2.0
    # outer site distance
    r_12 = np.sqrt(2.0) * L

    # dipolar length
    a_dd = (m * mu_0 * p ** 2) / (12 * np.pi * hbar ** 2)
    # scaterring length
    a = a_dd * 3 * np.sqrt(np.pi) / (4 * q ** 3)
    # lattice trap frequency
    omega = (2 * hbar * q ** 2) / (m * r_12 ** 2)
    # external trap frequency
    omega_ext = 3 * hbar / (m * L ** 2) * np.sqrt((a_dd / 2 / r_12) * (nu_bar / (1 + 2 * np.sqrt(3.0) * z_bar)))

    # on-site energy
    U_0 = (4 * np.pi * hbar ** 2 * a / m) * (q ** 2 / (r_12 ** 2) / np.pi) ** (1.5)
    # off-site energy
    U_12 = (3 * a_dd * hbar ** 2) / (m * r_12 ** 3)
    # interaction energy
    U = U_0 / 4.0

    # tunneling amplitude
    # Delta = 0.5 - 0.5 * (q / np.pi) ** 2 * (np.pi ** 2 / 2.0 - 3 + np.exp(-(np.pi / q) ** 2))
    Delta = 0.5 - (q / 2 / np.pi) ** 2 * (np.pi ** 2 - 3 + np.exp(-(np.pi / q) ** 2))
    J = -(hbar * q / r_12) ** 2 / m * np.exp(-q ** 2 / 4) * (1 + Delta)

    # interaction ratio
    U_ratio = U * (N - nu_bar / 2) / J


    # effective tunneling amplitude
    xi = 3 * J ** 2 / 4 / U / (N - 1 - nu_bar / 2.0)
    # protocol time
    tau = np.pi / xi

    # angular quasimomentum
    W = hbar * q ** 2 / 2 * np.exp(-q ** 2 / 2)
    # maximum angular speed
    Omega_max = xi / 3 / W

    # max rotational energy
    zeta_max = xi / 3
    # sensitivity f of zeta
    f = np.sqrt(2 + np.cos(np.pi * zeta_max / xi)) / np.cos(np.pi * zeta_max / xi / 2)
    # sensitivity
    Omega_dt = f / (2 * tau * W * np.sqrt(2 * N))


    # change parameter units
    lmbd = lmbd * 1e+9
    omega = omega * 1e-3 / 2.0 / np.pi # 2pi * KHz
    omega_ext = omega_ext * 1e-3 / 2.0 / np.pi # 2pi * KHz
    a_dd = a_dd / a_0 # Bohr radius
    p = p / mu_B # Bohr magneton
    a = a / a_0 # Bohr radius
    U_0 = U_0 / h # Hz
    U_12 = U_12 / h # Hz
    U = U / h # Hz
    J = J / h # Hz
    tau = tau * h # s
    Omega_max = Omega_max / 2.0 / np.pi # 2pi * Hz
    Omega_dt = Omega_dt / 2.0 / np.pi # 2pi * Hz
    dt_ratio = Omega_dt / Omega_max


    # organize results
    results = [
            ["Parameter q",             q,         ""      ],
            ["Number of particles",     N,         ""      ],
            ["Wave length",             lmbd,      "nm"    ],
            ["Trap frequency",          omega,     "2π KHz"],
            ["External trap frequency", omega_ext, "2π Hz" ],
            ["Magnetic moment",         p,         "μ_B"   ],
            ["Dipolar length",           a_dd,      "a_0"   ],
            ["Scattering length",       a,         "a_0"   ],
            ["On-site energy",          U_0,       "Hz"    ],
            ["Off-site energy",         U_12,      "Hz"    ],
            ["Interaction energy",      U,         "Hz"    ],
            ["Hopping rate",            J,         "Hz"    ],
            ["Interaction ratio",       U_ratio,   ""      ],
            ["Protocol time",           tau,       "s"     ],
            ["Max angular speed",       Omega_max, "2π Hz" ],
            ["Sensitivity",             Omega_dt,  "2π Hz" ],
            ["Sensitivity ratio",       dt_ratio,  ""      ],
        ]

    data_list = {
            "Parameter": [],
            "Value": [],
            "Units": [],
        }

    for result in results:
        param, value, units = result
        data_list["Parameter"].append(param)
        data_list["Value"].append(value)
        data_list["Units"].append(units)

    data = pd.DataFrame.from_dict(data_list)

    # build latex table
    latex_table = [
                    # ["Gaussian ratio", "$q$", f"{q:.3f}"],
                    # ["Number of particles", "$N$", f"{int(N)}"],
                    # ["Wave length", r"$\lambda$", f"{int(lmbd)} nm"],
                    ["Trap frequency", r"$\omega$", f"{omega:.2f} $2\\pi\\times$kHz"],
                    ["External trap frequency", r"$\omega_{\text{ext}}$", f"{omega_ext:.2f}" + r" $2\pi\times$kHz"],
                    ["Magnetic moment", "$p$", f"{p:.2f}" + r" $\mu_{B}$"],
                    ["Dipolar length", r"$a_{dd}$", f"{a_dd:.2f}" + r" $a_{0}$"],
                    ["Scattering length", "$a$", f"{a:.2f}" + r" $a_{0}$"],
                    # ["On-site energy", r"$U_{0}/h$", f"{U_0:.2f} Hz"],
                    # ["Off-site energy", r"$U_{12}/h$", f"{U_12:.2f} Hz"],
                    ["Interaction energy", r"$U/h$", f"{U:.2f} Hz"],
                    ["Hopping rate", r"$J/h$", f"{J:.2f} Hz"],
                    ["Interaction ratio", r"$\chi$", f"{U_ratio:.2f}"],
                    ["Protocol time", r"$\tau$", f"{tau:.2f} s"],
                    ["Max angular speed", r"$\Omega_{\text{max}}$", f"{Omega_max:.2f}" + r" $2\pi\times$Hz"],
                    ["Sensitivity", r"$\delta\Omega$", f"{Omega_dt:.2f}" + r" $2\pi\times$Hz"],
                    # ["Sensitivity ratio", r"$\delta\Omega/\Omega_{\text{max}}$", f"{dt_ratio:.2f}"],
                ]

    latex_list = {
            "Parameter": [],
            "Symbol": [],
            "Value": [],
        }
    
    for item in latex_table:
        param, symbol, Value = item
        latex_list["Parameter"].append(param)
        latex_list["Symbol"].append(symbol)
        latex_list["Value"].append(Value)

    latex_data = pd.DataFrame.from_dict(latex_list)

    return data, latex_data

Constants

In [3]:
# magnetic vacuum permittivity
mu_0 = sp.constants.mu_0
# plack's constant
h = sp.constants.h
# reduced plank's constant
hbar = sp.constants.hbar
# bohr radius
a_0 = sp.constants.physical_constants["Bohr radius"][0]
# bohr magneton
mu_B = sp.constants.physical_constants["Bohr magneton"][0]
# atomic mass constant
u = sp.constants.physical_constants["atomic mass constant"][0]

Experimental values for Dysprosium 164

In [4]:
# Parameters for Dysprosium 164
N = 16
q = 2.89
lmbd = 532.0 * 1e-9
m = 164 * u
p = mag_moment_from_dipole_length(131.97 * a_0, m)
nu_bar = 1.0
z_bar = 0.0

# Data calculation
data_dy_164, latex_dy_164 = calculate_experimental_parameters(N, q, lmbd, m, p, nu_bar, z_bar)
print("Experimental values for Dysprosium 164")
display(data_dy_164)

print(latex_dy_164.to_latex(index=False))

Experimental values for Dysprosium 164


,Parameter,Value,Units
0,Parameter q,2.890000,
1,Number of particles,16.000000,
2,Wave length,532.000000,nm
3,Trap frequency,7.275046,2π KHz
4,External trap frequency,0.251760,2π Hz
5,Magnetic moment,9.973806,μ_B
6,Dipolar length,131.970000,a_0
7,Scattering length,7.268050,a_0
8,On-site energy,24.255546,Hz
9,Off-site energy,24.255546,Hz


\begin{tabular}{lll}
\toprule
Parameter & Symbol & Value \\
\midrule
Trap frequency & $\omega$ & 7.28 $2\pi\times$kHz \\
External trap frequency & $\omega_{\text{ext}}$ & 0.25 $2\pi\times$kHz \\
Magnetic moment & $p$ & 9.97 $\mu_{B}$ \\
Dipolar length & $a_{dd}$ & 131.97 $a_{0}$ \\
Scattering length & $a$ & 7.27 $a_{0}$ \\
Interaction energy & $U/h$ & 6.06 Hz \\
Hopping rate & $J/h$ & 8.22 Hz \\
Interaction ratio & $\chi$ & 11.43 \\
Protocol time & $\tau$ & 5.45 s \\
Max angular speed & $\Omega_{\text{max}}$ & 3.00 $2\pi\times$Hz \\
Sensitivity & $\delta\Omega$ & 0.46 $2\pi\times$Hz \\
\bottomrule
\end{tabular}



Experimental values for Chromium 52

In [5]:
# Parameters for Chromium 52
N = 16
q = 2.876
lmbd = 532.0 * 1e-9
m = 52 * u
p = 6 * mu_B
nu_bar = 1.0
z_bar = 0.0

# Data calculation
data_ch_52, latex_ch_52 = calculate_experimental_parameters(N, q, lmbd, m, p, nu_bar, z_bar)
print("Experimental values for Chromium 52")
display(data_ch_52)

print(latex_ch_52.to_latex(index=False))

Experimental values for Chromium 52


,Parameter,Value,Units
0,Parameter q,2.876000,
1,Number of particles,16.000000,
2,Wave length,532.000000,nm
3,Trap frequency,22.722617,2π KHz
4,External trap frequency,0.268966,2π Hz
5,Magnetic moment,6.000000,μ_B
6,Dipolar length,15.143120,a_0
7,Scattering length,0.846223,a_0
8,On-site energy,8.777922,Hz
9,Off-site energy,8.777922,Hz


\begin{tabular}{lll}
\toprule
Parameter & Symbol & Value \\
\midrule
Trap frequency & $\omega$ & 22.72 $2\pi\times$kHz \\
External trap frequency & $\omega_{\text{ext}}$ & 0.27 $2\pi\times$kHz \\
Magnetic moment & $p$ & 6.00 $\mu_{B}$ \\
Dipolar length & $a_{dd}$ & 15.14 $a_{0}$ \\
Scattering length & $a$ & 0.85 $a_{0}$ \\
Interaction energy & $U/h$ & 2.19 Hz \\
Hopping rate & $J/h$ & 4.06 Hz \\
Interaction ratio & $\chi$ & 8.37 \\
Protocol time & $\tau$ & 8.07 s \\
Max angular speed & $\Omega_{\text{max}}$ & 1.96 $2\pi\times$Hz \\
Sensitivity & $\delta\Omega$ & 0.30 $2\pi\times$Hz \\
\bottomrule
\end{tabular}



Experimental values for Erbium 168

In [6]:
# Parameters for Erbium 168
N = 16
q = 2.886
lmbd = 532.0 * 1e-9
m = 168 * u
p = 7 * mu_B
nu_bar = 1.0
z_bar = 0.0

# Data calculation
data_er_168, latex_er_168 = calculate_experimental_parameters(N, q, lmbd, m, p, nu_bar, z_bar)
print("Experimental values for Erbium 168")
display(data_er_168)

print(latex_er_168.to_latex(index=False))

Experimental values for Erbium 168


,Parameter,Value,Units
0,Parameter q,2.886000,
1,Number of particles,16.000000,
2,Wave length,532.000000,nm
3,Trap frequency,7.082185,2π KHz
4,External trap frequency,0.174578,2π Hz
5,Magnetic moment,7.000000,μ_B
6,Dipolar length,66.590899,a_0
7,Scattering length,3.682664,a_0
8,On-site energy,11.947727,Hz
9,Off-site energy,11.947727,Hz


\begin{tabular}{lll}
\toprule
Parameter & Symbol & Value \\
\midrule
Trap frequency & $\omega$ & 7.08 $2\pi\times$kHz \\
External trap frequency & $\omega_{\text{ext}}$ & 0.17 $2\pi\times$kHz \\
Magnetic moment & $p$ & 7.00 $\mu_{B}$ \\
Dipolar length & $a_{dd}$ & 66.59 $a_{0}$ \\
Scattering length & $a$ & 3.68 $a_{0}$ \\
Interaction energy & $U/h$ & 2.99 Hz \\
Hopping rate & $J/h$ & 6.10 Hz \\
Interaction ratio & $\chi$ & 7.59 \\
Protocol time & $\tau$ & 4.87 s \\
Max angular speed & $\Omega_{\text{max}}$ & 3.32 $2\pi\times$Hz \\
Sensitivity & $\delta\Omega$ & 0.51 $2\pi\times$Hz \\
\bottomrule
\end{tabular}

